In [1]:
import re
import os
from datetime import datetime
from zoneinfo import ZoneInfo
from urllib.parse import urljoin, urlparse, parse_qs

import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
# 크롤링 타깃 및 스키마 설정

LISTING_URL = "https://www.makangs.com/search_list.php?keyword=%EC%84%9C%EC%9A%B8"
SOURCE_SITE = urlparse(LISTING_URL).netloc  # www.makangs.com

KST = ZoneInfo("Asia/Seoul")
CRAWLED_AT = datetime.now(KST).isoformat(timespec="seconds")

RAW_COLUMNS = [
    "source_site",
    "crawled_at",
    "listing_url",
    "detail_url",
    "external_id",
    "name_raw",
    "category_raw",
    "address_raw",
    "map_url_raw",
    "status_raw",
]

In [3]:
session = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
}

resp = session.get(LISTING_URL, headers=headers, timeout=30)
resp.raise_for_status()

html = resp.text
len(html)

991246

In [8]:
# 카드 파싱 
soup = BeautifulSoup(html, "html.parser")

cards = soup.select("div.itm__card")
print("cards:", len(cards))

rows = []
for card in cards:
    a = card.select_one("a[href]")
    if not a:
        continue

    detail_url = urljoin(LISTING_URL, a.get("href").strip())
    qs = parse_qs(urlparse(detail_url).query)
    external_id = qs.get("code", [None])[0]

    # title text (예: "역삼동 왁싱 라꾸왁싱")
    tit_el = card.select_one("strong.tit")
    tit_text = tit_el.get_text(" ", strip=True) if tit_el else None

    name_raw = None
    category_raw = None

    if tit_text:
        toks = [t for t in re.split(r"\s+", tit_text.strip()) if t]

        # name_raw: 마지막 토큰 = 업소명
        if toks:
            name_raw = toks[-1]

        # category_raw: 맨 앞(동) + 맨 뒤(업소명) 제외한 중간 토큰
        if len(toks) >= 3:
            category_raw = " ".join(toks[1:-1])
        else:
            category_raw = None

    # address (raw, include 안내문)
    addr_el = card.select_one("p.locate span")
    address_raw = addr_el.get_text(" ", strip=True) if addr_el else None

    rows.append({
        "source_site": SOURCE_SITE,
        "crawled_at": CRAWLED_AT,
        "listing_url": LISTING_URL,
        "detail_url": detail_url,
        "external_id": external_id,
        "name_raw": name_raw,
        "category_raw": category_raw,      
        "address_raw": address_raw,
        "map_url_raw": detail_url, # 지도 근거 URL 없으니 상세 URL로 대체
        "status_raw": None,        # “있다는 것=정상영업” 가정 → None
    })

df = pd.DataFrame(rows, columns=RAW_COLUMNS)
df.head(10)

cards: 334


,source_site,crawled_at,listing_url,detail_url,external_id,name_raw,category_raw,address_raw,map_url_raw,status_raw
0,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=5434,5434,데이지에스테틱,마사지 1인샵,서울특별시 송파구 백제고분로17길 36 / 잠실동 210-4 (1층 101호) / ...,https://www.makangs.com/shop.php?code=5434,None
1,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=2378,2378,고트테라피,마사지,서울특별시 서초구 신반포로47길 23 / 잠원동 39-5 / 논현역 7번 출구 인근,https://www.makangs.com/shop.php?code=2378,None
2,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=2772,2772,베스파,마사지,서울 종로구 삼일대로17길 51 / 관철동 180 / 종각역 4번 출구 도보 2분,https://www.makangs.com/shop.php?code=2772,None
3,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=5720,5720,라일락,마사지 1인샵,서울특별시 서초구 서초동 (상세주소 문의) / 강남역 5번 출구 도보 3분,https://www.makangs.com/shop.php?code=5720,None
4,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=5651,5651,당근테라피,마사지,서울특별시 영등포구 영중로10길 18 / 영등포동3가 16-1 (지하 2층) / 영...,https://www.makangs.com/shop.php?code=5651,None
5,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=5450,5450,차이테라피,마사지,서울특별시 송파구 백제고분로48길 29 / 방이동 142-16 / 송파나루역 3번 ...,https://www.makangs.com/shop.php?code=5450,None
6,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=5030,5030,도쿄센슈얼,마사지,서울특별시 강남구 역삼동 (상세주소 문의) / 강남역 인근,https://www.makangs.com/shop.php?code=5030,None
7,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=5387,5387,써니스웨디시,마사지,서울특별시 마포구 와우산로18길 12 / 서교동 360-21 / 홍대 클럽거리 인근,https://www.makangs.com/shop.php?code=5387,None
8,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=4094,4094,보희살롱,마사지,서울특별시 강남구 테헤란로13길 65 / 역삼동 623-22 301호,https://www.makangs.com/shop.php?code=4094,None
9,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=4608,4608,스파엔,마사지,서울특별시 강남구 봉은사로30길 73 / 역삼동 669-9 / 역삼역 7번 출구 도...,https://www.makangs.com/shop.php?code=4608,None


In [9]:
# 품질 체크
print("rows:", len(df))
print("external_id null:", df["external_id"].isna().sum())
print("detail_url null:", df["detail_url"].isna().sum())
print("name_raw null:", df["name_raw"].isna().sum())
print("address_raw null:", df["address_raw"].isna().sum())

# external_id가 없는 케이스는 detail_url 기준으로만 유니크 처리(merge 대상에서 제외)할 것

rows: 334
external_id null: 0
detail_url null: 0
name_raw null: 0
address_raw null: 0


In [10]:
def first_non_null(series: pd.Series):
    for v in series:
        if pd.notna(v) and v != "":
            return v
    return None

def join_unique(series: pd.Series, sep=";"):
    vals = []
    for v in series.dropna().astype(str).tolist():
        v = v.strip()
        if v and v not in vals:
            vals.append(v)
    return sep.join(vals) if vals else None

with_id = df[df["external_id"].notna()].copy()
no_id = df[df["external_id"].isna()].copy()

merged_with_id = (
    with_id
    .groupby("external_id", as_index=False)
    .agg({
        "source_site": first_non_null,
        "crawled_at": first_non_null,
        "listing_url": join_unique,
        "detail_url": first_non_null,     # external_id가 같으면 사실상 동일 상세
        "name_raw": first_non_null,
        "category_raw": first_non_null,   # 현재 None이므로 변화 없음
        "address_raw": first_non_null,
        "map_url_raw": first_non_null,
        "status_raw": first_non_null,
    })
)

df_final = pd.concat([merged_with_id, no_id[RAW_COLUMNS]], ignore_index=True)
df_final = df_final[RAW_COLUMNS]

print("before:", len(df), "after:", len(df_final))
df_final.head(10)

before: 334 after: 334


,source_site,crawled_at,listing_url,detail_url,external_id,name_raw,category_raw,address_raw,map_url_raw,status_raw
0,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=129,129,홍대입구,마사지,서울특별시 마포구 서교동 (상세주소 문의) / 홍대입구역 1번 출구 도보 1분,https://www.makangs.com/shop.php?code=129,None
1,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1407,1407,논현왓포타이마사지,마사지,서울 강남구 논현동 182-25번지 화랑닭발 4층,https://www.makangs.com/shop.php?code=1407,None
2,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1446,1446,두바이테라피,마사지,서울특별시 금천구 벚꽃로40길 3 / 가산동 42-16 / 가산디지털단지역 2번 출...,https://www.makangs.com/shop.php?code=1446,None
3,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1456,1456,타이365,마사지,서울 서대문구 연세로 4길 19 3층,https://www.makangs.com/shop.php?code=1456,None
4,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1490,1490,풋앤조이,마사지,서울 은평구 연서로211 하경빌딩 4층,https://www.makangs.com/shop.php?code=1490,None
5,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1497,1497,사월왁싱,왁싱,서울특별시 강남구 역삼로3길 17-6,https://www.makangs.com/shop.php?code=1497,None
6,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1499,1499,시온타이,마사지,"서울 종로구 종로10길14, 4층",https://www.makangs.com/shop.php?code=1499,None
7,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1501,1501,왁싱맨,왁싱,서울특별시 강남구 강남대로84길 23,https://www.makangs.com/shop.php?code=1501,None
8,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1504,1504,이모꾸비,왁싱,서울특별시 은평구 진흥로 17 역촌동 43-47 2층,https://www.makangs.com/shop.php?code=1504,None
9,www.makangs.com,2026-01-27T11:19:04+09:00,https://www.makangs.com/search_list.php?keywor...,https://www.makangs.com/shop.php?code=1507,1507,업클로즈왁싱,왁싱,서울특별시 강남구 삼성로71길 32,https://www.makangs.com/shop.php?code=1507,None


In [12]:
# CSV 저장
out_dir = os.path.abspath(os.path.join("..", "data"))
os.makedirs(out_dir, exist_ok=True)

ts = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(out_dir, f"makangs_raw_{ts}.csv")

df_final.to_csv(out_path, index=False, encoding="utf-8-sig")
out_path

'c:\\DSJA\\2025-sbsDSJA\\TeamProject\\data\\makangs_raw_20260127_113115.csv'